In [1]:
# =============================================================================
# Gold Layer Transformations — Microsoft Fabric Lakehouse
# =============================================================================
# Paste each CELL block into its own Fabric notebook cell.
#
# HOW NAMESPACING WORKS IN FABRIC:
#   - Fabric's Spark catalog only supports single-part table names
#     (e.g. "fact_crane_event"), NOT "lakehouse.dbo.table".
#   - To read from a NON-default lakehouse (Silver), use its ABFS path.
#   - To write to the Gold lakehouse (your notebook's DEFAULT lakehouse),
#     use saveAsTable("table_name") — single part only.
#
# SETUP BEFORE RUNNING:
#   1. Set the GOLD lakehouse as the DEFAULT lakehouse in the notebook sidebar.
#   2. Fill in SILVER_ABFS_ROOT below with your Silver lakehouse ABFS path.
#      You can find it in Fabric: Silver Lakehouse → ... → Properties → ABFS path
#      It looks like:
#      abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id>/Tables
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, FloatType, StringType, DateType

# -----------------------------------------------------------------------------
# CONFIG — set your Silver lakehouse ABFS root path here
# -----------------------------------------------------------------------------
SILVER_ABFS_ROOT = "abfss://POC@onelake.dfs.fabric.microsoft.com/silver_lh.Lakehouse/Tables"
# Example:
# SILVER_ABFS_ROOT = "abfss://abc123@onelake.dfs.fabric.microsoft.com/def456/Tables"

def silver(table_name: str):
    """Read a table from the Silver lakehouse via its ABFS Delta path."""
    return spark.read.format("delta").load(f"{SILVER_ABFS_ROOT}/{table_name}")

def write_gold(df, table_name: str, partition_cols: list = None):
    """
    Write a DataFrame as a managed Delta table into the Gold lakehouse.
    Gold must be set as the notebook's DEFAULT lakehouse in the sidebar.
    Single-part table name only — Fabric Spark catalog requirement.
    """
    writer = (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.saveAsTable(table_name)   # single-part — lands in default lakehouse
    print(f"✓ Written → {table_name}  ({df.count():,} rows)")


# =============================================================================
# CELL 1 — Vessel Call Performance Mart
# =============================================================================
# Grain  : one row per VesselCallID
# Sources: fact_crane_event  +  dim_crane
# Metrics: total moves, moves/hr, crane utilisation %, operation duration hrs
# =============================================================================

crane_events = silver("fact_crane_event")
dim_crane    = silver("dim_crane")

vessel_perf = (
    crane_events
    .join(dim_crane, on="CraneID", how="left")
    .groupBy("VesselCallID", "Terminal")
    .agg(
        F.sum("MovesCompleted").alias("TotalMovesCompleted"),
        F.sum("ActiveMinutes").alias("TotalActiveMinutes"),
        F.sum("DowntimeMinutes").alias("TotalDowntimeMinutes"),
        F.countDistinct("CraneID").alias("CranesDeployed"),
        F.min("EventTime").alias("OperationStart"),
        F.max("EventTime").alias("OperationEnd"),
    )
    .withColumn(
        "MovesPerHour",
        F.round(F.col("TotalMovesCompleted") / (F.col("TotalActiveMinutes") / 60.0), 2)
    )
    .withColumn(
        "CraneUtilisationPct",
        F.round(
            F.col("TotalActiveMinutes") * 100.0 /
            (F.col("TotalActiveMinutes") + F.col("TotalDowntimeMinutes")), 2
        )
    )
    .withColumn(
        "OperationDurationHours",
        F.round(
            (F.unix_timestamp("OperationEnd") - F.unix_timestamp("OperationStart"))
            / 3600.0, 2
        )
    )
    .withColumn(
        "OperationDateKey",
        F.date_format("OperationStart", "yyyyMMdd").cast(IntegerType())
    )
    .withColumn("LoadDate", F.current_date())
)

write_gold(vessel_perf, "gold_vessel_call_performance", partition_cols=["Terminal"])


# =============================================================================
# CELL 2 — Container Dwell Time Mart
# =============================================================================
# Grain  : one row per ContainerID
# Sources: fact_container_lifecycle  +  dim_importer  +  dim_yard_block
# Metrics: dwell hours per lifecycle stage, customs hold hrs, total dwell hrs
# =============================================================================

lc           = silver("fact_container_lifecycle")
dim_importer = silver("dim_importer")
dim_yard     = silver("dim_yard_block")

container_dwell = (
    lc
    .join(dim_importer, on="ImporterID",  how="left")
    .join(dim_yard,     on="YardBlockID", how="left")

    # Stage 1: Vessel discharge → yard storage
    .withColumn(
        "UnloadToYardHrs",
        F.round(
            (F.unix_timestamp("YardInTime") - F.unix_timestamp("ArrivalTime"))
            / 3600.0, 2
        )
    )
    # Stage 2: Yard in → customs submission
    .withColumn(
        "YardToCustomsSubmitHrs",
        F.round(
            (F.unix_timestamp("CustomsSubmitTime") - F.unix_timestamp("YardInTime"))
            / 3600.0, 2
        )
    )
    # Stage 3: Customs submission → approval (customs hold)
    .withColumn(
        "CustomsHoldHrs",
        F.when(
            F.col("CustomsApproveTime").isNotNull(),
            F.round(
                (F.unix_timestamp("CustomsApproveTime") -
                 F.unix_timestamp("CustomsSubmitTime")) / 3600.0, 2
            )
        ).otherwise(F.lit(None).cast(FloatType()))
    )
    # Stage 4: Customs approval → yard out (pickup wait)
    .withColumn(
        "ApprovalToPickupHrs",
        F.when(
            F.col("YardOutTime").isNotNull() & F.col("CustomsApproveTime").isNotNull(),
            F.round(
                (F.unix_timestamp("YardOutTime") -
                 F.unix_timestamp("CustomsApproveTime")) / 3600.0, 2
            )
        ).otherwise(F.lit(None).cast(FloatType()))
    )
    # Stage 5: Yard out → gate exit
    .withColumn(
        "YardOutToExitHrs",
        F.when(
            F.col("ExitTime").isNotNull() & F.col("YardOutTime").isNotNull(),
            F.round(
                (F.unix_timestamp("ExitTime") - F.unix_timestamp("YardOutTime"))
                / 3600.0, 2
            )
        ).otherwise(F.lit(None).cast(FloatType()))
    )
    # Total dwell: arrival → gate exit
    .withColumn(
        "TotalDwellHrs",
        F.when(
            F.col("ExitTime").isNotNull(),
            F.round(
                (F.unix_timestamp("ExitTime") - F.unix_timestamp("ArrivalTime"))
                / 3600.0, 2
            )
        ).otherwise(F.lit(None).cast(FloatType()))
    )
    .withColumn("IsCustomsCleared", F.col("CustomsApproveTime").isNotNull())
    .withColumn("IsExited",         F.col("Status") == "Exited")
    .withColumn(
        "ArrivalDateKey",
        F.date_format("ArrivalTime", "yyyyMMdd").cast(IntegerType())
    )
    .withColumn("LoadDate", F.current_date())
    .select(
        "ContainerID", "VesselCallID",
        "ImporterID", "ImporterName", "Industry",
        "YardBlockID", "Terminal", "Zone",
        "ArrivalTime", "YardInTime", "CustomsSubmitTime",
        "CustomsApproveTime", "YardOutTime", "ExitTime",
        "Status", "IsCustomsCleared", "IsExited",
        "UnloadToYardHrs", "YardToCustomsSubmitHrs",
        "CustomsHoldHrs", "ApprovalToPickupHrs",
        "YardOutToExitHrs", "TotalDwellHrs",
        "ArrivalDateKey", "LoadDate"
    )
)

write_gold(container_dwell, "gold_container_dwell_time", partition_cols=["Terminal"])


# =============================================================================
# CELL 3 — Crane Productivity Mart
# =============================================================================
# Grain  : one row per CraneEventID
# Sources: fact_crane_event  +  dim_crane
# Metrics: moves/hr, utilisation %, downtime ratio, crane age at event date
# =============================================================================

crane_events = silver("fact_crane_event")
dim_crane    = silver("dim_crane")

crane_prod = (
    crane_events
    .join(dim_crane, on="CraneID", how="left")
    .withColumn(
        "MovesPerHour",
        F.round(F.col("MovesCompleted") / (F.col("ActiveMinutes") / 60.0), 2)
    )
    .withColumn(
        "UtilisationPct",
        F.round(
            F.col("ActiveMinutes") * 100.0 /
            (F.col("ActiveMinutes") + F.col("DowntimeMinutes")), 2
        )
    )
    .withColumn(
        "DowntimeRatio",
        F.round(
            F.col("DowntimeMinutes") /
            (F.col("ActiveMinutes") + F.col("DowntimeMinutes")), 4
        )
    )
    .withColumn(
        "CraneAgeYearsAtEvent",
        F.round(
            F.datediff(F.col("EventTime").cast(DateType()),
                       F.col("CommissionDate")) / 365.25, 1
        )
    )
    .withColumn(
        "EventDateKey",
        F.date_format("EventTime", "yyyyMMdd").cast(IntegerType())
    )
    .withColumn("LoadDate", F.current_date())
    .select(
        "CraneEventID", "CraneID", "CraneType", "Terminal",
        "CommissionDate", "CraneAgeYearsAtEvent",
        "VesselCallID", "EventTime", "EventDateKey", "ShiftName",
        "ActiveMinutes", "DowntimeMinutes", "MovesCompleted",
        "MovesPerHour", "UtilisationPct", "DowntimeRatio",
        "LoadDate"
    )
)

write_gold(crane_prod, "gold_crane_productivity", partition_cols=["Terminal"])


# =============================================================================
# CELL 4 — Gate Throughput Mart
# =============================================================================
# Grain  : one row per TransactionID
# Sources: fact_gate_transaction  +  dim_gate  +  dim_truck
# Metrics: processing mins, peak hour flag, inbound/outbound split
# =============================================================================

txns      = silver("fact_gate_transaction")
dim_gate  = silver("dim_gate")
dim_truck = silver("dim_truck")

gate_throughput = (
    txns
    .join(dim_gate,  on="GateID",  how="left")
    .join(dim_truck, on="TruckID", how="left")
    .withColumn("EntryHour",      F.hour("EntryTime"))
    .withColumn("EntryDayOfWeek", F.dayofweek("EntryTime"))
    # Peak hours: 06:00–09:59 and 14:00–17:59
    .withColumn(
        "IsPeakHour",
        F.col("EntryHour").between(6, 9) | F.col("EntryHour").between(14, 17)
    )
    .withColumn(
        "TransactionDurationMins",
        F.round(
            (F.unix_timestamp("ExitTime") - F.unix_timestamp("EntryTime"))
            / 60.0, 2
        )
    )
    .withColumn(
        "EntryDateKey",
        F.date_format("EntryTime", "yyyyMMdd").cast(IntegerType())
    )
    .withColumn("LoadDate", F.current_date())
    .select(
        "TransactionID", "GateID", "GateType", "Terminal",
        "TruckID", "CarrierName", "VehicleType",
        "ContainerID", "Direction",
        "EntryTime", "ExitTime", "EntryDateKey",
        "EntryHour", "EntryDayOfWeek", "IsPeakHour",
        "ProcessingMinutes", "TransactionDurationMins",
        "LoadDate"
    )
)

write_gold(gate_throughput, "gold_gate_throughput", partition_cols=["Terminal"])


# =============================================================================
# CELL 5 — Equipment Maintenance Mart
# =============================================================================
# Grain  : one row per MaintenanceID
# Sources: fact_maintenance  +  dim_equipment
# Metrics: downtime hrs, cost/hr, days since last maintenance (MTBF proxy)
# =============================================================================

maint     = silver("fact_maintenance")
dim_equip = silver("dim_equipment")

w_mtbf = Window.partitionBy("EquipmentID").orderBy("StartTime")

equip_maint = (
    maint
    .join(dim_equip, on="EquipmentID", how="left")
    .withColumn("PrevMaintenanceEnd", F.lag("EndTime", 1).over(w_mtbf))
    .withColumn(
        "DaysSincePrevMaintenance",
        F.when(
            F.col("PrevMaintenanceEnd").isNotNull(),
            F.round(
                (F.unix_timestamp("StartTime") -
                 F.unix_timestamp("PrevMaintenanceEnd")) / 86400.0, 1
            )
        ).otherwise(F.lit(None).cast(FloatType()))
    )
    .withColumn(
        "CostPerDowntimeHour",
        F.round(F.col("Cost") / F.col("DowntimeHours"), 2)
    )
    .withColumn(
        "FailureCategory",
        F.when(F.col("FailureReason") == "Routine",            "Routine")
         .when(F.col("FailureReason") == "Mechanical Wear",    "Mechanical")
         .when(F.col("FailureReason") == "Electrical Failure", "Electrical")
         .otherwise("Other")
    )
    .withColumn(
        "MaintenanceDateKey",
        F.date_format("StartTime", "yyyyMMdd").cast(IntegerType())
    )
    .withColumn("LoadDate", F.current_date())
    .select(
        "MaintenanceID", "EquipmentID", "EquipmentType", "Terminal",
        "StartTime", "EndTime", "MaintenanceDateKey",
        "MaintenanceType", "FailureReason", "FailureCategory",
        "DowntimeHours", "Cost", "CostPerDowntimeHour",
        "PrevMaintenanceEnd", "DaysSincePrevMaintenance",
        "LoadDate"
    )
)

write_gold(equip_maint, "gold_equipment_maintenance",
           partition_cols=["Terminal", "MaintenanceType"])


# =============================================================================
# CELL 6 — Date Dimension
# =============================================================================
# Grain  : one row per calendar day
# Auto-derives date range from fact_container_lifecycle ArrivalTime
# =============================================================================

lc = silver("fact_container_lifecycle")

date_range = lc.select(
    F.min(F.to_date("ArrivalTime")).alias("min_date"),
    F.max(F.to_date("ArrivalTime")).alias("max_date")
).collect()[0]

min_date = str(date_range["min_date"])
max_date = str(date_range["max_date"])

dim_date = (
    spark.sql(f"""
        SELECT explode(
            sequence(date '{min_date}', date '{max_date}', interval 1 day)
        ) AS CalendarDate
    """)
    .withColumn("DateKey",     F.date_format("CalendarDate", "yyyyMMdd").cast(IntegerType()))
    .withColumn("Year",        F.year("CalendarDate"))
    .withColumn("Quarter",     F.quarter("CalendarDate"))
    .withColumn("Month",       F.month("CalendarDate"))
    .withColumn("MonthName",   F.date_format("CalendarDate", "MMMM"))
    .withColumn("WeekOfYear",  F.weekofyear("CalendarDate"))
    .withColumn("DayOfMonth",  F.dayofmonth("CalendarDate"))
    .withColumn("DayOfWeek",   F.dayofweek("CalendarDate"))     # 1=Sun, 7=Sat
    .withColumn("DayName",     F.date_format("CalendarDate", "EEEE"))
    .withColumn("IsWeekend",   F.col("DayOfWeek").isin([1, 7]))
    .withColumn("YearMonth",   F.date_format("CalendarDate", "yyyy-MM"))
    .withColumn("YearQuarter",
        F.concat(
            F.year("CalendarDate").cast(StringType()),
            F.lit("-Q"),
            F.quarter("CalendarDate").cast(StringType())
        )
    )
    .withColumn("LoadDate", F.current_date())
)

write_gold(dim_date, "gold_dim_date")


# =============================================================================
# CELL 7 — KPI Summary Table
# =============================================================================
# Grain  : one row per (CalendarDate × Terminal)
# Sources: the five gold marts written above (Cells 1–5)
# NOTE   : Run AFTER Cells 2, 3, 4, and 5 have completed successfully.
#
# Reads gold tables via single-part name — they are in the default lakehouse.
# =============================================================================

# Container KPIs
dwell_g = spark.read.table("gold_container_dwell_time")
container_kpis = (
    dwell_g
    .withColumn("CalendarDate", F.to_date("ArrivalTime"))
    .groupBy("CalendarDate", "Terminal")
    .agg(
        F.count("ContainerID").alias("ContainersArrived"),
        F.sum(F.when(F.col("IsExited"), 1).otherwise(0)).alias("ContainersExited"),
        F.round(F.avg("TotalDwellHrs"),   2).alias("AvgDwellHrs"),
        F.round(F.avg("CustomsHoldHrs"),  2).alias("AvgCustomsHoldHrs"),
        F.round(F.avg("UnloadToYardHrs"), 2).alias("AvgUnloadToYardHrs"),
    )
)

# Crane KPIs
crane_g = spark.read.table("gold_crane_productivity")
crane_kpis = (
    crane_g
    .withColumn("CalendarDate", F.to_date("EventTime"))
    .groupBy("CalendarDate", "Terminal")
    .agg(
        F.round(F.avg("MovesPerHour"),   2).alias("AvgCraneMovesPerHour"),
        F.round(F.avg("UtilisationPct"), 2).alias("AvgCraneUtilisationPct"),
        F.sum("DowntimeMinutes").alias("TotalCraneDowntimeMins"),
        F.sum("MovesCompleted").alias("TotalCraneMoves"),
    )
)

# Gate KPIs
gate_g = spark.read.table("gold_gate_throughput")
gate_kpis = (
    gate_g
    .withColumn("CalendarDate", F.to_date("EntryTime"))
    .groupBy("CalendarDate", "Terminal")
    .agg(
        F.count("TransactionID").alias("TotalGateTxns"),
        F.round(F.avg("ProcessingMinutes"), 2).alias("AvgGateProcessingMins"),
        F.sum(F.when(F.col("IsPeakHour"),              1).otherwise(0)).alias("PeakHourTxns"),
        F.sum(F.when(F.col("Direction") == "Inbound",  1).otherwise(0)).alias("InboundTxns"),
        F.sum(F.when(F.col("Direction") == "Outbound", 1).otherwise(0)).alias("OutboundTxns"),
    )
)

# Maintenance KPIs
maint_g = spark.read.table("gold_equipment_maintenance")
maint_kpis = (
    maint_g
    .withColumn("CalendarDate", F.to_date("StartTime"))
    .groupBy("CalendarDate", "Terminal")
    .agg(
        F.sum("DowntimeHours").alias("TotalMaintenanceDowntimeHrs"),
        F.sum("Cost").alias("TotalMaintenanceCost"),
        F.count("MaintenanceID").alias("MaintenanceEvents"),
        F.sum(F.when(F.col("MaintenanceType") == "Corrective", 1).otherwise(0))
          .alias("CorrectiveEvents"),
    )
)

kpi_summary = (
    container_kpis
    .join(crane_kpis, on=["CalendarDate", "Terminal"], how="full")
    .join(gate_kpis,  on=["CalendarDate", "Terminal"], how="full")
    .join(maint_kpis, on=["CalendarDate", "Terminal"], how="full")
    .withColumn(
        "DateKey",
        F.date_format("CalendarDate", "yyyyMMdd").cast(IntegerType())
    )
    .withColumn("LoadDate", F.current_date())
    .orderBy("CalendarDate", "Terminal")
)

write_gold(kpi_summary, "gold_kpi_summary", partition_cols=["Terminal"])


# =============================================================================
# CELL 8 — SCD Type 2 Dimensions
# =============================================================================
# Adds EffectiveStart, EffectiveEnd, IsCurrent, RowHash, SurrogateKey.
# Full initial load — all rows marked as current.
#
# For incremental runs: compare RowHash against existing gold rows,
# expire changed rows (EffectiveEnd = today, IsCurrent = False),
# then insert new versions via MERGE INTO on the Delta table.
# =============================================================================

def build_scd2(silver_table: str, gold_table: str, attribute_cols: list):
    """
    silver_table   : table name as it exists under SILVER_ABFS_ROOT
    gold_table     : single-part name written to default (Gold) lakehouse
    attribute_cols : columns whose changes trigger a new SCD2 version
    """
    df = silver(silver_table)
    hash_input = [F.coalesce(F.col(c).cast(StringType()), F.lit(""))
                  for c in attribute_cols]
    gold = (
        df
        .withColumn("RowHash",        F.md5(F.concat_ws("|", *hash_input)))
        .withColumn("EffectiveStart",  F.current_date())
        .withColumn("EffectiveEnd",    F.lit("9999-12-31").cast(DateType()))
        .withColumn("IsCurrent",       F.lit(True))
        .withColumn("SurrogateKey",    F.monotonically_increasing_id())
        .withColumn("LoadDate",        F.current_date())
    )
    write_gold(gold, gold_table)

# dim_vessel — tracks vessel name, shipping line, capacity, flag changes
build_scd2(
    silver_table   = "dim_vessel",
    gold_table     = "gold_dim_vessel_scd2",
    attribute_cols = ["VesselName", "ShippingLineID", "VesselType",
                      "TEUCapacity", "FlagCountry"]
)

# dim_importer — tracks importer name and industry changes
build_scd2(
    silver_table   = "dim_importer",
    gold_table     = "gold_dim_importer_scd2",
    attribute_cols = ["ImporterName", "Industry"]
)

# dim_shipping_line — tracks alliance membership changes
build_scd2(
    silver_table   = "dim_shipping_line",
    gold_table     = "gold_dim_shipping_line_scd2",
    attribute_cols = ["ShippingLineName", "Alliance"]
)

print("\n✓ All gold tables built successfully.")

StatementMeta(, 9ca079d7-914c-473c-8b59-01a2bdab8ec7, 3, Finished, Available, Finished, False)

✓ Written → gold_vessel_call_performance  (1,434 rows)
✓ Written → gold_container_dwell_time  (20,000 rows)
✓ Written → gold_crane_productivity  (3,000 rows)
✓ Written → gold_gate_throughput  (15,000 rows)
✓ Written → gold_equipment_maintenance  (500 rows)
✓ Written → gold_dim_date  (709 rows)
✓ Written → gold_kpi_summary  (1,548 rows)
✓ Written → gold_dim_vessel_scd2  (30 rows)
✓ Written → gold_dim_importer_scd2  (50 rows)
✓ Written → gold_dim_shipping_line_scd2  (8 rows)

✓ All gold tables built successfully.
